In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import glob
print(glob.glob("**/olist_orders_dataset.csv", recursive=True))
print(glob.glob("/kaggle/input/**/olist_orders_dataset.csv", recursive=True))

[]
['/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv']


In [4]:
import os
import pandas as pd

D = "/kaggle/input/datasets/olistbr/brazilian-ecommerce/"
print(os.listdir(D))

['olist_customers_dataset.csv', 'olist_sellers_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'olist_geolocation_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_payments_dataset.csv']


In [5]:
orders = pd.read_csv(
    D + "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                 "order_estimated_delivery_date"])
customers = pd.read_csv(D + "olist_customers_dataset.csv",
                        usecols=["customer_id", "customer_unique_id"])
items = pd.read_csv(D + "olist_order_items_dataset.csv",
                    usecols=["order_id", "seller_id", "price", "freight_value"])
pay = pd.read_csv(D + "olist_order_payments_dataset.csv",
                  usecols=["order_id", "payment_value"])
print("loaded")

loaded


In [6]:
orders = pd.read_csv(
    D + "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                 "order_estimated_delivery_date"])
customers = pd.read_csv(D + "olist_customers_dataset.csv",
                        usecols=["customer_id", "customer_unique_id"])
items = pd.read_csv(D + "olist_order_items_dataset.csv",
                    usecols=["order_id", "seller_id", "price", "freight_value"])
pay = pd.read_csv(D + "olist_order_payments_dataset.csv",
                  usecols=["order_id", "payment_value"])
print("loaded")

loaded


In [8]:
for name in ["D", "orders", "customers", "items", "pay", "df", "delivered", "m"]:
    print(name, name in globals())

D True
orders True
customers True
items True
pay True
df False
delivered False
m False


In [9]:
order_value = pay.groupby("order_id")["payment_value"].sum().rename("order_value")
df = orders.merge(customers, on="customer_id", how="left")
df = df.merge(order_value, on="order_id", how="left")
df["week"] = df["order_purchase_timestamp"].dt.to_period("W-SUN").dt.start_time
delivered = df[df["order_status"] == "delivered"].copy()

m = df.groupby("week").agg(orders_placed=("order_id", "nunique"))
m["orders_delivered"] = delivered.groupby("week")["order_id"].nunique()
m["fulfilled_order_value"] = delivered.groupby("week")["order_value"].sum().round(2)
m["fulfilment_rate_pct"] = (m["orders_delivered"] / m["orders_placed"] * 100).round(1)

first_week = df.groupby("customer_unique_id")["week"].min()
df["is_repeat"] = df["week"] > df["customer_unique_id"].map(first_week)
m["active_buyers"] = df.groupby("week")["customer_unique_id"].nunique()
m["repeat_buyers"] = df[df["is_repeat"]].groupby("week")["customer_unique_id"].nunique()
m["repeat_buyer_rate_pct"] = (m["repeat_buyers"] / m["active_buyers"] * 100).round(2)
print("done")

done


In [10]:
sellers = items.merge(delivered[["order_id", "week"]], on="order_id")
m["active_sellers"] = sellers.groupby("week")["seller_id"].nunique()
seller_rev = sellers.groupby(["week", "seller_id"])["price"].sum()
top = seller_rev.groupby("week").max() / seller_rev.groupby("week").sum() * 100
m["top_seller_share_pct"] = top.round(1)

d = delivered.dropna(subset=["order_delivered_customer_date"]).copy()
d["delivery_days"] = (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.days
d["on_time"] = d["order_delivered_customer_date"] <= d["order_estimated_delivery_date"]
m["on_time_delivery_pct"] = (d.groupby("week")["on_time"].mean() * 100).round(1)
m["avg_delivery_days"] = d.groupby("week")["delivery_days"].mean().round(1)
print("done")

done


In [11]:
m = m.fillna(0).reset_index()
m = m[(m["week"] >= "2017-01-02") & (m["week"] <= "2018-08-20")]
m.to_csv("/kaggle/working/weekly_metrics.csv", index=False)
m.tail(10)

,week,orders_placed,orders_delivered,fulfilled_order_value,fulfilment_rate_pct,active_buyers,repeat_buyers,repeat_buyer_rate_pct,active_sellers,top_seller_share_pct,on_time_delivery_pct,avg_delivery_days
84,2018-06-18,1457,1443.0,225424.93,99.0,1453,52.0,3.58,567.0,2.9,98.8,8.0
85,2018-06-25,1401,1382.0,219584.94,98.6,1395,37.0,2.65,581.0,2.9,99.3,7.9
86,2018-07-02,1196,1178.0,184784.27,98.5,1193,33.0,2.77,520.0,2.2,98.4,8.1
87,2018-07-09,1001,972.0,180520.25,97.1,996,26.0,2.61,461.0,6.1,93.9,10.8
88,2018-07-16,1678,1649.0,258226.48,98.3,1651,51.0,3.09,635.0,2.6,96.1,8.6
89,2018-07-23,1646,1610.0,278926.28,97.8,1636,46.0,2.81,671.0,2.8,96.1,7.9
90,2018-07-30,2058,2002.0,320204.74,97.3,2052,65.0,3.17,736.0,2.6,86.2,8.2
91,2018-08-06,1988,1931.0,318700.65,97.1,1981,52.0,2.62,711.0,3.8,87.4,8.0
92,2018-08-13,1875,1839.0,285766.68,98.1,1867,62.0,3.32,721.0,2.5,92.8,7.2
93,2018-08-20,1071,1054.0,145680.87,98.4,1069,41.0,3.84,525.0,4.3,93.3,5.0


In [12]:
print(m.shape)
print(m["week"].min(), m["week"].max())

(86, 12)
2017-01-02 00:00:00 2018-08-20 00:00:00


In [13]:
full = pd.date_range(m["week"].min(), m["week"].max(), freq="7D")
print("missing weeks:", len(set(full) - set(m["week"])))

missing weeks: 0


In [14]:
print(len(df), df["order_id"].nunique())
print(len(delivered))
print(round(delivered["order_value"].sum(), 2))

99441 99441
96478
15422461.77


In [15]:
w = delivered[(delivered["week"] >= "2017-01-02") & (delivered["week"] <= "2018-08-20")]
print(round(w["order_value"].sum(), 2), round(m["fulfilled_order_value"].sum(), 2))
print(w["order_id"].nunique(), m["orders_delivered"].sum())

15363879.91 15363879.91
96095 96095.0


In [16]:
pct = [c for c in m.columns if c.endswith("_pct")]
print(m[pct].describe().loc[["min", "max"]])

     fulfilment_rate_pct  repeat_buyer_rate_pct  top_seller_share_pct  \
min                 91.2                   0.00                   2.0   
max                 99.3                   4.32                  31.2   

     on_time_delivery_pct  
min                  71.0  
max                  99.6  


In [17]:
wk = m["week"].iloc[20]
x = df[df["week"] == wk]
print(wk, x["order_id"].nunique(), m.loc[m["week"] == wk, "orders_placed"].iloc[0])

2017-05-22 00:00:00 784 784


In [18]:
print(df["customer_unique_id"].nunique(), df["customer_id"].nunique())
print(m["repeat_buyer_rate_pct"].describe())

96096 99441
count    86.000000
mean      2.017209
std       0.961755
min       0.000000
25%       1.420000
50%       2.140000
75%       2.642500
max       4.320000
Name: repeat_buyer_rate_pct, dtype: float64
